# Hypotes – geografins påverkan på bostadspriset

Den här notebooken undersöker om geografiska variabler förbättrar modellens förmåga att uppskatta utgångspriset.

Vi jämför två versioner av samma modell:

1. En modell utan kommun, latitud och longitud.
2. Den befintliga globalmodellen med geografiska variabler.

Modellerna använder samma tränings- och testbostäder. Den huvudsakliga skillnaden är därför om geografiska features ingår.

Notebooken skapar även två filer som senare används av Streamlit:

- `data/geography_metrics.json`
- `data/municipality_profiles.csv`


## Importera paket

Vi använder samma scikit-learn-komponenter som i modellträningen. Den färdigtränade globalmodellen laddas med Joblib.


In [41]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## Sökvägar

Notebooken förutsätter att den ligger i projektets `notebooks`-mapp.


In [42]:
DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")

CLEANED_DATA_PATH = DATA_DIR / "cleaned_housing_data.parquet"
GLOBAL_MODEL_PATH = MODELS_DIR / "global_model.joblib"
METRICS_PATH = DATA_DIR / "geography_metrics.json"
PROFILES_PATH = DATA_DIR / "municipality_profiles.csv"


## Läs den tvättade datan

Datan har redan tvättats i `model_training.ipynb`. På så sätt slipper vi duplicera reglerna för datatvätt här.


In [43]:
if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(
        "Kör Parquet-exporten i model_training.ipynb först. "
        f"Filen saknas: {CLEANED_DATA_PATH}"
    )

if not GLOBAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Den sparade globalmodellen saknas. "
        f"Filen saknas: {GLOBAL_MODEL_PATH}"
    )

df = pd.read_parquet(CLEANED_DATA_PATH)

print(f"Antal tvättade bostäder: {len(df)}")
df.head()


Antal tvättade bostäder: 10090


,ad_id,date_published,typology,asking_price_sek,land_area_sqm,living_area_sqm,sqm_price_sek,number_rooms,address,location,coordenates,latitude,longitude,municipality,has_land_area
0,21440984,2024-12-28,APARTMENT,3695000.0,NaN,82.0,45061.0,4.0,Einar Hansens esplanad 14,"Västra Hamnen, Malmö kommun","55.6118431,12.9822884",55.611843,12.982288,Malmö kommun,0
1,21462070,2024-12-28,APARTMENT,3095000.0,NaN,109.0,28394.0,4.0,Skarpskyttevägen 30A,"Norra Fäladen, Lunds kommun","55.7230072,13.1973028",55.723007,13.197303,Lunds kommun,0
2,21455084,2024-12-28,APARTMENT,2295000.0,NaN,54.0,42500.0,2.0,Boplatsvägen 9,"Brotorp/järvastaden, Sundbybergs kommun","59.381943,17.973993",59.381943,17.973993,Sundbybergs kommun,0
3,21411619,2024-12-28,APARTMENT,3675000.0,NaN,119.0,30882.0,4.0,Drottninggatan 34C,"Alingsås, Alingsås kommun","57.93111,12.5369",57.931110,12.536900,Alingsås kommun,0
4,21461458,2024-12-28,APARTMENT,295000.0,NaN,99.0,2980.0,4.0,Valhallagatan 19A,"Skara, Skara kommun","58.38283,13.4190693",58.382830,13.419069,Skara kommun,0


## Återskapa samma datauppdelning

Samma `random_state`, teststorlek och stratifiering används som i modellträningen. Därför får hypotesen samma train-, validation- och testbostäder.


In [44]:
train_validation_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["typology"]
)

train_df, validation_df = train_test_split(
    train_validation_df,
    test_size=0.25,
    random_state=42,
    stratify=train_validation_df["typology"]
)

final_train_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

print(f"Slutlig träningsdata: {len(final_train_df)}")
print(f"Testdata: {len(test_df)}")


Slutlig träningsdata: 8072
Testdata: 2018


## Funktion för utvärdering

Vi använder MAE, RMSE, medianfel och R², precis som i modellträningen.


In [45]:
def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "Median error": median_absolute_error(y_true, predictions),
        "R2": r2_score(y_true, predictions)
    }


## Ladda modellen med geografi

Globalmodellen innehåller redan kommun, latitud och longitud. Vi återanvänder både den färdigtränade pipelinen och dess metadata.


In [46]:
global_bundle = joblib.load(GLOBAL_MODEL_PATH)

global_model = global_bundle["pipeline"]
global_metadata = global_bundle["metadata"]

with_geography_features = global_metadata["feature_columns"]

print("Modell:", global_metadata["model_name"])
print("Features med geografi:", with_geography_features)


Modell: HistGradientBoostingRegressor
Features med geografi: ['land_area_sqm', 'living_area_sqm', 'number_rooms', 'latitude', 'longitude', 'has_land_area', 'municipality', 'typology']


## Välj features utan geografi

Modellen utan geografi får fortfarande information om bostadens storlek, antal rum, tomt och bostadstyp. Kommun och koordinater tas bort.


In [47]:
without_geography_numeric_features = [
    "land_area_sqm",
    "living_area_sqm",
    "number_rooms",
    "has_land_area"
]

without_geography_categorical_features = [
    "typology"
]

without_geography_features = (
    without_geography_numeric_features
    + without_geography_categorical_features
)

without_geography_features


['land_area_sqm',
 'living_area_sqm',
 'number_rooms',
 'has_land_area',
 'typology']

## Preprocessing utan geografi

Numeriska saknade värden fylls med medianen. Bostadstypen one-hot-kodas.


In [48]:
without_geography_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        without_geography_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ),
        without_geography_categorical_features
    )
])


## Träna modellen utan geografi

Vi klonar estimatorn från den färdiga globalmodellen. Därmed återanvänds samma modelltyp och hyperparametrar. Bara preprocessing och feature-listan skiljer sig.


In [49]:
without_geography_model = Pipeline([
    (
        "preprocessor",
        without_geography_preprocessor
    ),
    (
        "model",
        clone(global_model.named_steps["model"])
    )
])

without_geography_model.fit(
    final_train_df[without_geography_features],
    final_train_df["asking_price_sek"]
)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['land_area_sqm','living_area_sqm','number_rooms','has_land_area', 'typology']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This 

## Gör prediktioner på samma testdata

Ingen av modellerna tränas på testdatan. Båda utvärderas på exakt samma bostäder.


In [50]:
without_geography_predictions = without_geography_model.predict(
    test_df[without_geography_features]
)

with_geography_predictions = global_model.predict(
    test_df[with_geography_features]
)

without_geography_metrics = calculate_metrics(
    test_df["asking_price_sek"],
    without_geography_predictions
)

with_geography_metrics = calculate_metrics(
    test_df["asking_price_sek"],
    with_geography_predictions
)


## Jämför resultaten

Lägre MAE och RMSE är bättre. Högre R² är bättre.


In [51]:
geography_comparison = pd.DataFrame([
    {
        "Model": "Utan geografi",
        **without_geography_metrics
    },
    {
        "Model": "Med geografi",
        **with_geography_metrics
    }
])

geography_comparison


,Model,MAE,RMSE,Median error,R2
0,Utan geografi,1.173126e+06,1.604466e+06,866298.254006,0.255812
1,Med geografi,6.206929e+05,9.147439e+05,401603.505934,0.758109


## Beräkna förbättringen

En positiv procentsats innebär att modellen med geografi har lägre RMSE.


In [52]:
rmse_without_geography = without_geography_metrics["RMSE"]
rmse_with_geography = with_geography_metrics["RMSE"]

rmse_improvement_percent = (
    (rmse_without_geography - rmse_with_geography)
    / rmse_without_geography
    * 100
)

print(
    "RMSE utan geografi:",
    f"{rmse_without_geography:,.0f} kr"
)

print(
    "RMSE med geografi:",
    f"{rmse_with_geography:,.0f} kr"
)

print(
    "Förbättring:",
    f"{rmse_improvement_percent:.1f} %"
)


RMSE utan geografi: 1,604,466 kr
RMSE med geografi: 914,744 kr
Förbättring: 43.0 %


## Resultat per bostadstyp

Den här tabellen visar om geografin hjälper olika mycket för lägenheter, villor och radhus.


In [53]:
segment_results = []

for typology in ["APARTMENT", "HOUSE", "ROW_HOUSE"]:
    segment_mask = test_df["typology"] == typology
    segment_target = test_df.loc[
        segment_mask,
        "asking_price_sek"
    ]

    metrics_without = calculate_metrics(
        segment_target,
        without_geography_predictions[segment_mask]
    )

    metrics_with = calculate_metrics(
        segment_target,
        with_geography_predictions[segment_mask]
    )

    segment_improvement = (
        (metrics_without["RMSE"] - metrics_with["RMSE"])
        / metrics_without["RMSE"]
        * 100
    )

    segment_results.append({
        "Bostadstyp": typology,
        "RMSE utan geografi": metrics_without["RMSE"],
        "RMSE med geografi": metrics_with["RMSE"],
        "Förbättring (%)": segment_improvement
    })

segment_comparison = pd.DataFrame(segment_results)
segment_comparison


,Bostadstyp,RMSE utan geografi,RMSE med geografi,Förbättring (%)
0,APARTMENT,1.409738e+06,7.092119e+05,49.691919
1,HOUSE,2.104607e+06,1.364085e+06,35.185738
2,ROW_HOUSE,1.566952e+06,9.453018e+05,39.672590


## Skapa kommunprofiler

Varje kombination av kommun och bostadstyp får representativa koordinater, antal observationer och medianpris per kvadratmeter.

Kommuner med få observationer sparas fortfarande i filen. Streamlit-sidan kan senare kräva minst 20 observationer innan en kommun visas.


In [54]:
profile_data = df.copy()

profile_data["calculated_sqm_price"] = (
    profile_data["asking_price_sek"]
    / profile_data["living_area_sqm"]
)

municipality_profiles = (
    profile_data
    .groupby(
        ["municipality", "typology"],
        as_index=False
    )
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        observations=("asking_price_sek", "size"),
        median_asking_price=("asking_price_sek", "median"),
        median_sqm_price=("calculated_sqm_price", "median")
    )
    .sort_values(
        ["typology", "municipality"]
    )
)

municipality_profiles.head()


,municipality,typology,latitude,longitude,observations,median_asking_price,median_sqm_price
0,Ale kommun,APARTMENT,57.847958,12.023709,12,1985000.0,30468.112573
3,Alingsås kommun,APARTMENT,57.928396,12.530906,50,2175000.0,31160.714286
6,Alvesta kommun,APARTMENT,56.899468,14.554268,1,425000.0,9550.561798
8,Aneby kommun,APARTMENT,57.832610,14.814540,1,520000.0,6582.278481
10,Arboga kommun,APARTMENT,59.391230,15.840200,2,935000.0,11675.523838


## Kvalitetskontroll av kommunprofiler

Vi kontrollerar att kommunprofilerna saknar tomma värden och dubletter samt att koordinater och priser är rimliga.

Profiler med mycket få bostäder kan ge instabila jämförelser. Appen kommer därför endast erbjuda kombinationer av kommun och bostadstyp som har minst 10 observationer. Alla profiler behålls ändå i CSV-filen.


In [55]:
MIN_OBSERVATIONS = 10

print("Antal kommunprofiler:", len(municipality_profiles))

print("\nBostadstyper:")
print(
    municipality_profiles["typology"]
    .value_counts()
)

print("\nSaknade värden:")
print(
    municipality_profiles.isna().sum()
)

print("\nAntal duplicerade kommunprofiler:")
print(
    municipality_profiles.duplicated(
        subset=["municipality", "typology"]
    ).sum()
)

reliable_profiles = municipality_profiles[
    municipality_profiles["observations"]
    >= MIN_OBSERVATIONS
].copy()

print("\nGodkända kommunprofiler:", len(reliable_profiles))
print(
    reliable_profiles["typology"]
    .value_counts()
)

assert municipality_profiles["municipality"].notna().all()
assert municipality_profiles["typology"].isin(
    ["APARTMENT", "HOUSE", "ROW_HOUSE"]
).all()
assert municipality_profiles["latitude"].between(
    55.0, 69.1
).all()
assert municipality_profiles["longitude"].between(
    10.5, 24.2
).all()
assert (municipality_profiles["observations"] > 0).all()
assert (municipality_profiles["median_asking_price"] > 0).all()
assert (municipality_profiles["median_sqm_price"] > 0).all()
assert not municipality_profiles.duplicated(
    subset=["municipality", "typology"]
).any()

print("\nAlla kvalitetskontroller godkända.")


Antal kommunprofiler: 594

Bostadstyper:
typology
HOUSE        261
APARTMENT    215
ROW_HOUSE    118
Name: count, dtype: int64

Saknade värden:
municipality           0
typology               0
latitude               0
longitude              0
observations           0
median_asking_price    0
median_sqm_price       0
dtype: int64

Antal duplicerade kommunprofiler:
0

Godkända kommunprofiler: 179
typology
APARTMENT    90
HOUSE        74
ROW_HOUSE    15
Name: count, dtype: int64

Alla kvalitetskontroller godkända.


## Spara resultat till appen

Streamlit behöver bara läsa de färdiga filerna. Appen behöver därför inte träna modeller eller bearbeta hela datasetet vid uppstart.


In [56]:
def json_metrics(metrics):
    return {
        "mae": float(metrics["MAE"]),
        "rmse": float(metrics["RMSE"]),
        "median_error": float(metrics["Median error"]),
        "r2": float(metrics["R2"])
    }

metrics_to_save = {
    "without_geography": json_metrics(
        without_geography_metrics
    ),
    "with_geography": json_metrics(
        with_geography_metrics
    ),
    "rmse_improvement_percent": float(
        rmse_improvement_percent
    ),
    "test_rows": int(len(test_df)),
    "random_state": 42,
    "segments": {
        row["Bostadstyp"]: {
            "rmse_without_geography": float(
                row["RMSE utan geografi"]
            ),
            "rmse_with_geography": float(
                row["RMSE med geografi"]
            ),
            "improvement_percent": float(
                row["Förbättring (%)"]
            )
        }
        for row in segment_results
    }
}

with METRICS_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        metrics_to_save,
        file,
        ensure_ascii=False,
        indent=2
    )

municipality_profiles.to_csv(
    PROFILES_PATH,
    index=False,
    encoding="utf-8"
)

print(f"Sparad: {METRICS_PATH}")
print(f"Sparad: {PROFILES_PATH}")


Sparad: ..\data\geography_metrics.json
Sparad: ..\data\municipality_profiles.csv


## Slutsats

Modellen med geografiska variabler uppnådde ett RMSE på cirka 915 000 kronor, jämfört med cirka 1 604 000 kronor utan geografi. Det motsvarar en förbättring på cirka 43 procent.

Geografin förbättrade modellens resultat för samtliga bostadstyper:

- Lägenheter: cirka 50 procent lägre RMSE
- Villor: cirka 35 procent lägre RMSE
- Radhus: cirka 40 procent lägre RMSE

Resultatet ger starkt stöd för hypotesen att kommun och koordinater bidrar till modellens förmåga att uppskatta bostäders utgångspris.

Resultatet visar däremot ett prediktivt samband och bevisar inte att geografin ensam orsakar prisskillnaderna. Datasetet saknar exempelvis information om bostadens skick, våningsplan, månadsavgift, närhet till kollektivtrafik, skolor och annan lokal service.